In [156]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import xgboost as xgb
from sklearn.metrics import f1_score
import joblib
import explainer
import warnings
warnings.filterwarnings("ignore")

import importlib
importlib.reload(explainer)

<module 'explainer' from 'c:\\Users\\HP\\Desktop\\PhD Algorithm\\Notebooks\\explainer.py'>

In [169]:
# UNSW-NB15 dataset
test = pd.read_csv('../Datasets/UNSW-NB15/processed/le_test.csv').drop(['id','attack_cat'], axis=1)
train_cmc = pd.read_csv('../Datasets/UNSW-NB15/processed/train_cmc.csv').drop(['id','attack_cat'], axis=1)
model = joblib.load('../Models/UNSW-NB15.pkl')
analysis_features = ['label', 'confidence_score', 'confidence_type', 'predicted', 'CMC']

In [165]:
# KDD99 dataset
train_cmc = pd.read_csv('../Datasets/KDD99/train_cmc.csv')
X_test = pd.read_csv('../Datasets/KDD99/X_test.csv')
y_test = pd.read_csv('../Datasets/KDD99/y_test.csv')
test = X_test.copy()
test['label'] = y_test
model = joblib.load('../Models/KDD99.pkl')
analysis_features = ['label', 'predicted', 'CMC']

In [ ]:
def get_95_rules(cmc, analysis_features):
    len_rows = cmc.shape[0]
    confidence_rules_list = []

    for feature in tqdm(cmc.columns.drop(analysis_features)):
        # Get CMC distribution for each feature value
        value_cmc_counts = cmc.groupby(feature)['CMC'].value_counts(normalize=True).unstack().fillna(0)

        # Find feature values that have a single dominant group with at least 95% confidence
        dominant_values = value_cmc_counts[(value_cmc_counts.max(axis=1) >= 0.95)]

        if dominant_values.empty:
            continue

        # Get dominant class for each feature value
        dominant_class = dominant_values.idxmax(axis=1)

        # Get IDs for all dominant values
        id_map = cmc[cmc[feature].isin(dominant_values.index)].groupby(feature).apply(lambda x: x.index.tolist())

        # Construct rules
        for val in dominant_values.index:
            group = dominant_class[val]
            ids = id_map[val]

            # Assign predictions based on group type
            if group in ['TP', 'TN']:
                pred = 1 if group == 'TP' else 0
            elif group in ['FP', 'FN']:
                pred = 0 if group == 'FP' else 1  # FP -> 0, FN -> 1
            else:
                continue  # Skip if the group is unexpected

            support = len(ids)
            pct = round(len(ids) / len_rows * 100, 2)
            confidence = round(dominant_values.loc[val, group] * 100, 2)

            if support > 2:  # Filter out low-support rules
                confidence_rules_list.append([feature, '==', val, pred, support, pct, confidence, group, ids])

    return pd.DataFrame(confidence_rules_list, columns=['feature', 'is', 'value', 'predict', 'support', 'pct', 'confidence', 'group', 'ids'])

In [170]:
high_confidence_rules = get_95_rules(train_cmc, analysis_features)

100%|██████████| 42/42 [00:54<00:00,  1.29s/it]


In [180]:
importlib.reload(explainer)
explainer.apply_exclusive_rules(test, high_confidence_rules[high_confidence_rules['confidence'] == 100], model) 

  0%|          | 0/9381 [00:00<?, ?it/s]

100%|██████████| 9381/9381 [00:06<00:00, 1386.78it/s]


Test set coverage: 99.72%
Test coverage F1-score: 0.92
--------------------------------------
Total Skipped: 499 (0.28%)
 - Due to Conflicts: 0 (0.0%)
 - Due to No Rule Coverage: 499 (0.28%)
--------------------------------------
Using model to predict non-covered cases.
F1-score (if using entire dataset): 0.9177


In [181]:
explainer.apply_exclusive_rules(test, high_confidence_rules[high_confidence_rules['confidence'] == 100]) 

100%|██████████| 9381/9381 [00:07<00:00, 1245.64it/s]


Test set coverage: 99.72%
Test coverage F1-score: 0.92
--------------------------------------
Total Skipped: 499 (0.28%)
 - Due to Conflicts: 0 (0.0%)
 - Due to No Rule Coverage: 499 (0.28%)
--------------------------------------
Using Majority class to predict non-covered cases.
F1-score (if using entire dataset): 0.9182
